# Projet de Credit Scoring — Contexte bancaire ivoirien

## Objectif métier
Construire un modèle expérimental d'aide à l'analyse du risque de défaut pour des demandes de crédit de particuliers. Le projet suit une démarche adaptée au contexte bancaire ivoirien ; les données d'entraînement proviennent de **Kaggle** et ne sont pas des données de clients d'une banque ivoirienne.

**Question Data :** les caractéristiques disponibles dans le dataset permettent-elles de distinguer les dossiers associés à un défaut (`loan_status = 1`) des autres (`loan_status = 0`) ?

**Limite importante :** l'horizon exact associé à `loan_status` n'étant pas établi ici, nous ne qualifierons pas la cible de défaut à 12 mois. Les résultats sont pédagogiques et ne valident pas un déploiement bancaire.

## 1. Importer les bibliothèques

In [ ]:
# Manipulation et analyse des données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Préparation et modélisation
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, precision_score, recall_score, f1_score
)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Bibliothèques importées avec succès.')

## 2. Charger les données depuis Kaggle
Nous utilisons le dataset `credit_risk_dataset.csv` publié sur Kaggle. L'installation de `kagglehub` permet de charger le fichier directement dans Colab.

In [ ]:
# Si nécessaire dans Colab, décommentez la ligne suivante :
# %pip install -q kagglehub

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'nicholas33/credit-risk-dataset-csv',
    'credit_risk_dataset.csv'
)

print(f'Nombre de lignes : {df.shape[0]:,}')
print(f'Nombre de colonnes : {df.shape[1]}')
display(df.head())

## 3. Audit initial de la qualité des données
Cette étape vérifie la structure, les types, les valeurs manquantes, les doublons et la distribution de la cible avant toute modélisation.

In [ ]:
# Structure et statistiques descriptives
df.info()
display(df.describe(include='all').T)

# Valeurs manquantes
missing = pd.DataFrame({
    'nombre_manquant': df.isna().sum(),
    'pourcentage_manquant': (df.isna().mean() * 100).round(2)
}).query('nombre_manquant > 0')
display(missing)

# Doublons exacts
print('Nombre de doublons exacts :', df.duplicated().sum())

# Distribution de la cible
display(df['loan_status'].value_counts().rename_axis('loan_status').to_frame('effectif'))
display((df['loan_status'].value_counts(normalize=True) * 100).round(2).rename('pourcentage'))

## 4. Analyse exploratoire ciblée
On examine les variables financières et les taux de défaut observés dans le dataset. Une association statistique ne prouve pas une relation causale.

In [ ]:
# Comparaison de quelques variables numériques selon la cible
variables_numeriques = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_percent_income', 'cb_person_cred_hist_length'
]
display(df.groupby('loan_status')[variables_numeriques].median().T)

# Taux de défaut par catégorie pour quelques variables
for variable in ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']:
    taux = df.groupby(variable, dropna=False)['loan_status'].mean().sort_values(ascending=False)
    print(f'\nTaux de défaut par {variable} :')
    display((taux * 100).round(2).rename('taux_defaut_pct').to_frame())

# Visualisation du ratio montant du prêt / revenu
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='loan_percent_income', hue='loan_status', bins=30,
             stat='density', common_norm=False, element='step')
plt.title('Distribution du ratio prêt/revenu selon le statut')
plt.xlabel('Ratio prêt/revenu')
plt.ylabel('Densité')
plt.tight_layout()
plt.show()

## 5. Définir la cible et les variables explicatives
Nous excluons `loan_grade` et `loan_int_rate` de cette première version, car leur moment de disponibilité et leur rôle dans le processus d'octroi ne sont pas suffisamment établis ici. Cette précaution limite le risque d'utiliser une information issue d'une décision déjà prise.

In [ ]:
# Cible observée dans le dataset : 0 = non-défaut, 1 = défaut
target = 'loan_status'

# Variables retenues pour l'expérimentation
features = [
    'person_age',
    'person_income',
    'person_home_ownership',
    'person_emp_length',
    'loan_intent',
    'loan_amnt',
    'loan_percent_income',
    'cb_person_default_on_file',
    'cb_person_cred_hist_length'
]

# Retirer les doublons exacts et ne conserver que les colonnes utiles
df_model = df.drop_duplicates().copy()
df_model = df_model[features + [target]].copy()

print('Dimensions après suppression des doublons :', df_model.shape)
display(df_model.head())

## 6. Traiter les valeurs aberrantes et séparer les données
Les valeurs d'âge supérieures à 100 ans et les anciennetés professionnelles supérieures à 60 ans sont traitées comme valeurs suspectes pour cet exercice. Ces seuils sont des choix pratiques d'analyse, pas des seuils réglementaires. La séparation train/test est faite avant l'imputation afin d'apprendre les paramètres de préparation uniquement sur le train.

In [ ]:
# Transformer les valeurs jugées manifestement improbables en valeurs manquantes
df_model.loc[df_model['person_age'] > 100, 'person_age'] = np.nan
df_model.loc[df_model['person_emp_length'] > 60, 'person_emp_length'] = np.nan

X = df_model[features].copy()
y = df_model[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train :', X_train.shape, '| Test :', X_test.shape)
print('Taux de défaut train :', round(y_train.mean() * 100, 2), '%')
print('Taux de défaut test :', round(y_test.mean() * 100, 2), '%')

## 7. Prétraitement sans fuite de données
Le pipeline impute les valeurs manquantes à partir du train, encode les catégories et standardise les variables numériques pour la régression logistique. Le prétraitement est intégré au modèle pour appliquer exactement les mêmes transformations aux données de test et aux nouvelles données.

In [ ]:
numeric_features = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_percent_income', 'cb_person_cred_hist_length'
]
categorical_features = [
    'person_home_ownership', 'loan_intent', 'cb_person_default_on_file'
]

# Pipeline numérique : imputation par médiane puis standardisation
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline catégoriel : imputation de la modalité la plus fréquente puis One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 8. Modèle de référence : régression logistique
La régression logistique sert de modèle de référence. Nous évaluons les probabilités et les prédictions sur le jeu de test, en gardant à l'esprit que le seuil par défaut est 0,50.

In [ ]:
modele_logistique = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=3000, random_state=42))
])

modele_logistique.fit(X_train, y_train)
probas_log = modele_logistique.predict_proba(X_test)[:, 1]
pred_log = (probas_log >= 0.50).astype(int)

print('Accuracy :', round(accuracy_score(y_test, pred_log), 4))
print('AUC-ROC :', round(roc_auc_score(y_test, probas_log), 4))
print('\nMatrice de confusion :\n', confusion_matrix(y_test, pred_log))
print('\nRapport de classification :\n', classification_report(
    y_test, pred_log, target_names=['Sans défaut (0)', 'Défaut (1)'], digits=4
))

## 9. Modèle candidat : Random Forest
La Random Forest peut capturer des relations non linéaires. `class_weight='balanced'` augmente le poids de la classe minoritaire pendant l'apprentissage. Les métriques sont comparées au même seuil de référence ; le seuil de 0,40 utilisé lors de l'expérimentation précédente doit être sélectionné sur une validation séparée avant toute évaluation finale impartiale.

In [ ]:
# Le même prétraitement est utilisé, sans standardisation nécessaire pour les arbres.
numeric_transformer_rf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor_rf = ColumnTransformer(transformers=[
    ('num', numeric_transformer_rf, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

modele_rf = Pipeline(steps=[
    ('preprocessing', preprocessor_rf),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

modele_rf.fit(X_train, y_train)
probas_rf = modele_rf.predict_proba(X_test)[:, 1]

# Seuil de démonstration ; il doit être sélectionné sur validation en usage rigoureux.
seuil_rf = 0.40
pred_rf = (probas_rf >= seuil_rf).astype(int)

print('AUC-ROC :', round(roc_auc_score(y_test, probas_rf), 4))
print('Accuracy :', round(accuracy_score(y_test, pred_rf), 4))
print('Rappel des défauts :', round(recall_score(y_test, pred_rf), 4))
print('Précision des défauts :', round(precision_score(y_test, pred_rf, zero_division=0), 4))
print('F1-score des défauts :', round(f1_score(y_test, pred_rf, zero_division=0), 4))
print('\nMatrice de confusion :\n', confusion_matrix(y_test, pred_rf))
print('\nRapport de classification :\n', classification_report(
    y_test, pred_rf, target_names=['Sans défaut (0)', 'Défaut (1)'], digits=4
))

## 10. Importance des variables
L'importance des variables des arbres est une mesure interne du modèle : elle ne démontre ni causalité ni équité. Les variables catégorielles sont encodées en plusieurs colonnes ; on les agrège ici par variable d'origine pour faciliter la lecture.

In [ ]:
# Récupérer les noms des colonnes après prétraitement
feature_names = modele_rf.named_steps['preprocessing'].get_feature_names_out()
importances = modele_rf.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'variable_encodee': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

display(importance_df.head(10))

plt.figure(figsize=(10, 6))
top10 = importance_df.head(10).sort_values('importance')
plt.barh(top10['variable_encodee'], top10['importance'])
plt.title('Top 10 des variables encodées — Random Forest')
plt.xlabel('Importance relative')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

## 11. Conclusion métier

- La Random Forest est un modèle candidat intéressant dans cette expérimentation si l'objectif est d'augmenter le rappel des défauts.
- Une hausse du rappel peut augmenter les faux positifs ; le seuil doit être choisi selon les coûts des erreurs et l'objectif opérationnel.
- Les variables importantes doivent être vérifiées au regard de leur disponibilité au moment de la demande, de leur qualité et des risques de biais.
- Les données proviennent de Kaggle et ne représentent pas une clientèle bancaire ivoirienne. Les résultats ne peuvent pas être transposés directement à une banque locale.
- Le projet constitue un prototype pédagogique, pas un modèle validé pour décider automatiquement de l'octroi d'un crédit.

### Suites possibles
1. Choisir le seuil sur un ensemble de validation distinct.
2. Examiner la calibration des probabilités et les coûts des erreurs.
3. Tester la stabilité, l'équité et l'explicabilité du modèle.
4. Documenter le modèle, les limites et les conditions d'une éventuelle validation métier.